A localization or mapping filter has to score each [LIDAR](/aiml-common/lectures/sensor-models/lidar/index) scan: given a pose $s_t$ in a map $m$, how probable is the scan the sensor returned? The beam model of *Probabilistic Robotics* (Thrun, Burgard and Fox, Section 6.3) first predicts the range each beam should read. It then allows the measured range to differ from that prediction in four ways: a small measurement error, an object the map does not contain, no return at all, and a random return.

You define the four distributions, combine them into the likelihood of a scan, connect their parameters to the datasheet of the TurtleBot3's LDS-01, test them on a simulated room, and estimate the parameters from range readings.

## Expected range by ray casting

The prediction comes from the map. Place the sensor at its pose in the map and follow a straight line from it in the direction of beam $k$ until the line meets the first wall or occupied cell. The length of that line is the *expected range* $\hat z_t^k$, the reading a noise-free sensor would return if the map were complete. Following such a line through the map is called *ray casting*. In an occupancy grid it steps from cell to cell along the line and stops at the first occupied cell. A line that meets nothing within the sensor's maximum range $z_{\max}$ gives $\hat z_t^k = z_{\max}$.

The line starts at the sensor, not at the robot's center. With the robot at $s_t = (x, y, \theta)$ and the sensor mounted at $(x_s, y_s, \theta_s)$ in the robot frame, the sensor sits at

$$
\begin{bmatrix} x + x_s\cos\theta - y_s\sin\theta \\ y + x_s\sin\theta + y_s\cos\theta \end{bmatrix}
$$

in the map, and beam $k$, at bearing $\phi_k$ in the sensor frame, points at the angle $\theta + \theta_s + \phi_k$. The notation for a scan is:

- Scan: $\mathbf z_t = \{ z_t^k \}_{k=1}^K$, one range per beam.
- Expected range: $\hat z_t^k = r(s_t, \phi_k, m)$, where $r$ casts the ray of beam $k$ from the sensor pose above.
- Range limits of the sensor: $z_{\min}$ and $z_{\max}$.

## The four components

Each component is a distribution of the measured range $z$ of one beam, given its expected range $\hat z$ (Thrun, Burgard and Fox, Section 6.3.1). Thrun, Burgard and Fox define them on $[0, z_{\max}]$. A real sensor cannot report a range below its minimum $z_{\min}$, so here the continuous components live on $[z_{\min}, z_{\max})$, and the value $z_{\max}$ itself is kept for a beam that returned nothing.

*Hit.* A return from the surface the map predicts, with Gaussian measurement noise:

$$
p_{\text{hit}}(z \mid \hat z) = \eta_{\text{hit}}\, \mathcal N(z;\, \hat z, \sigma_{\text{hit}}^2) \quad \text{for } z_{\min} \le z < z_{\max},
$$

where $\eta_{\text{hit}}$ makes the density integrate to one over that interval.

*Short.* A return from an object between the sensor and the predicted surface, such as a person the map does not contain. Nearer objects block the beam first, so the density falls with range:

$$
p_{\text{short}}(z \mid \hat z) = \frac{\lambda_{\text{short}}\, e^{-\lambda_{\text{short}} z}}{e^{-\lambda_{\text{short}} z_{\min}} - e^{-\lambda_{\text{short}} \hat z}} \quad \text{for } z_{\min} \le z \le \hat z.
$$

*Max.* A beam that returned nothing reads $z_{\max}$. This component is a point mass, $P_{\max}(z = z_{\max}) = 1$. Thrun, Burgard and Fox write it as the indicator $I(z = z_{\max})$ and note that it has no density (Eq. 6.11).

*Random.* A reading that none of the other components explains, spread evenly over the range:

$$
p_{\text{rand}}(z) = \frac{1}{z_{\max} - z_{\min}} \quad \text{for } z_{\min} \le z < z_{\max}.
$$

Each component is zero outside its interval. The model mixes them with weights $\alpha_{\text{hit}}, \alpha_{\text{short}}, \alpha_{\max}, \alpha_{\text{rand}} \ge 0$ that sum to one (Thrun, Burgard and Fox write them $z_{\text{hit}}, \dots$; Eq. 6.13). Because one component is a point mass, the mixture has two parts. A reading below $z_{\max}$ has the density

$$
p(z \mid \hat z) = \alpha_{\text{hit}}\, p_{\text{hit}}(z \mid \hat z) + \alpha_{\text{short}}\, p_{\text{short}}(z \mid \hat z) + \alpha_{\text{rand}}\, p_{\text{rand}}(z), \qquad z_{\min} \le z < z_{\max},
$$

which integrates to $1 - \alpha_{\max}$, and a reading of exactly $z_{\max}$ has probability $\alpha_{\max}$.

The model assumes that each reading comes from a single ray. That suits a LIDAR's narrow beam better than a sonar, whose echo can come from anywhere in a cone of about 15° ([SensComp](https://www.senscomp.com/wp-content/uploads/2025/06/SensComp-PID-604142-Series-600-Instrument-Grade-Transducer-Spec.pdf)).

## The likelihood of a scan

The likelihood of beam $k$ is the mixture evaluated at its own reading and expected range,

$$
p(z_t^k \mid s_t, m) = p\big(z_t^k \mid \hat z_t^k\big), \qquad \hat z_t^k = r(s_t, \phi_k, m).
$$

The model treats the beams as conditionally independent given the pose and the map, so the likelihood of a scan is the product over its beams (Table 6.1). A computer adds logarithms instead, since a product of hundreds of small numbers underflows:

$$
\log p(\mathbf z_t \mid s_t, m) = \sum_{k=1}^K \log p\big(z_t^k \mid \hat z_t^k\big).
$$

The beams are not really independent. Neighboring beams hit the same surface, and an error in the pose or the map moves many of their predictions together. The product counts that shared evidence many times over, which makes the likelihood overconfident. Implementations therefore use only some of the beams, or scale down the summed log likelihood.

## Model parameters from the datasheet

The datasheet supplies the range limits, the angular sampling and the scan rate. A calibration of the mounting supplies the sensor pose, and recorded scans supply the mixture weights and the noise parameters. The table maps each entry to its symbol, with the values of the TurtleBot3's [ROBOTIS LDS-01](https://emanual.robotis.com/docs/en/platform/turtlebot3/appendix_lds_01/).

| Entry | LDS-01 | Symbol | Where it enters the model |
|---|---|---|---|
| Scan angle and angular resolution (datasheet) | 360° in 1° steps | $K = 360$ beams at bearings $\phi_k$ | ray casting along $\phi_k$ gives the expected range $\hat z_t^k$ |
| Maximum range (datasheet) | 3.5 m | $z_{\max}$ | a beam with no return reads $z_{\max}$; the continuous components end there |
| Minimum range (datasheet) | 0.12 m | $z_{\min}$ | the continuous components start there |
| Precision and accuracy (datasheet) | ±3.5% and ±5% beyond 0.5 m | starting value for $\sigma_{\text{hit}}$ | the width of $p_{\text{hit}}$ around $\hat z_t^k$ |
| Rotation rate (datasheet) | 300 ± 10 rpm | about 0.2 s per scan | how often the filter receives a measurement update |
| Mounting (calibration) | measured on the robot | sensor pose $(x_s, y_s, \theta_s)$ in the robot frame | where each ray starts |
| Recorded scans | depends on the scene | $\alpha_{\text{hit}}, \alpha_{\text{short}}, \alpha_{\max}, \alpha_{\text{rand}}, \lambda_{\text{short}}, \sigma_{\text{hit}}$ | estimated as in the last part of this section |

A ROS driver reports the range limits with each scan, as `range_min` and `range_max` of the `LaserScan` message ([sensor_msgs/LaserScan](https://docs.ros2.org/latest/api/sensor_msgs/msg/LaserScan.html)). [REP 117](https://www.ros.org/reps/rep-0117.html) asks drivers to report a beam with no return as $+\infty$, an invalid reading as NaN and a reading too close to measure as $-\infty$. Before scoring a scan, map $+\infty$ to $z_{\max}$, so that the max component scores it, and drop the NaN and $-\infty$ readings. A beam with no return carries information: no surface within $z_{\max}$ reflected the pulse along it.

The datasheet gives two error figures. Precision usually describes how much repeated readings of the same surface scatter. Accuracy also includes a systematic offset. $p_{\text{hit}}$ is centered on the true range, so it models the scatter, and an offset is better removed by calibration. Neither figure is a standard deviation, since the datasheet quotes a bound and does not say how many standard deviations it covers. The code below reads the ±3.5% as two standard deviations, an illustrative choice that gives $\sigma_{\text{hit}} \approx 0.0175\, \hat z$, or 3.5 cm at 2 m. That standard deviation grows with the expected range. Thrun's $p_{\text{hit}}$ has one $\sigma_{\text{hit}}$ for all ranges, so a LIDAR specified in percent calls for $\sigma_{\text{hit}}(\hat z)$ in its place. The fit at the end of this section estimates it from readings instead.

The code takes the datasheet values, derives $K$, the bearings $\phi_k$, the range limits and $\sigma_{\text{hit}}(\hat z)$, and evaluates the mixture for expected ranges of 1 m and 3 m. The mixture weights are illustrative.

In [ ]:
%config InlineBackend.figure_formats = ["svg"]

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Circle, Rectangle

plt.rcParams.update({"figure.figsize": (6.5, 3.5), "font.size": 9, "axes.titlesize": 10, "legend.fontsize": 8,
                     "svg.fonttype": "path", "axes.grid": False})
BLUE, VERMILLION, GREEN, PINK, GOLD, GRAY = "#0072B2", "#D55E00", "#009E73", "#cb77a6", "#c28600", "#7f7f7f"

In [ ]:
from scipy.special import ndtr  # standard normal CDF

# ROBOTIS LDS-01 datasheet values
lds01 = {"scan_angle_deg": 360, "resolution_deg": 1.0, "range_min": 0.12, "range_max": 3.5,
         "precision_near": 0.010, "precision_frac": 0.035, "rotation_rpm": 300}

K = int(lds01["scan_angle_deg"] / lds01["resolution_deg"])  # number of beams
phi = np.deg2rad(np.arange(K) * lds01["resolution_deg"])  # beam bearings in the sensor frame
z_min, z_max = lds01["range_min"], lds01["range_max"]
FRAC = lds01["precision_frac"] / 2  # illustrative: the +-3.5% read as two standard deviations


def sigma_hit(z_hat, frac=FRAC):
    """Standard deviation of p_hit: 5 mm below 0.5 m, frac * z_hat beyond."""
    return np.where(z_hat < 0.5, lds01["precision_near"] / 2, frac * z_hat)


# Not on the datasheet: these depend on the scene and are estimated from readings.
alpha = {"hit": 0.85, "short": 0.08, "max": 0.04, "rand": 0.03}
lam_short = 1.0  # 1/m


def p_hit(z, z_hat, frac=FRAC):
    """Gaussian around z_hat, normalized over [z_min, z_max)."""
    s = sigma_hit(z_hat, frac)
    eta = 1 / (ndtr((z_max - z_hat) / s) - ndtr((z_min - z_hat) / s))
    dens = eta * np.exp(-0.5 * ((z - z_hat) / s) ** 2) / (s * np.sqrt(2 * np.pi))
    return np.where((z >= z_min) & (z < z_max), dens, 0.0)


def p_short(z, z_hat, lam=lam_short):
    """Exponential on [z_min, z_hat], for objects the map does not contain."""
    norm = np.exp(-lam * z_min) - np.exp(-lam * np.maximum(z_hat, z_min))
    with np.errstate(divide="ignore", invalid="ignore"):
        dens = lam * np.exp(-lam * z) / norm
    return np.where((z >= z_min) & (z <= z_hat) & (norm > 0), dens, 0.0)


def p_rand(z):
    """Uniform on [z_min, z_max)."""
    return np.where((z >= z_min) & (z < z_max), 1 / (z_max - z_min), 0.0)


def p_beam(z, z_hat, a=alpha, frac=FRAC, lam=lam_short):
    """Density of a reading z_min <= z < z_max. A reading of exactly z_max has probability a["max"] instead."""
    return a["hit"] * p_hit(z, z_hat, frac) + a["short"] * p_short(z, z_hat, lam) + a["rand"] * p_rand(z)


def log_lik_beam(z, z_hat, a=alpha, frac=FRAC, lam=lam_short):
    """Log likelihood of each reading: log a["max"] at z_max, the log density below it."""
    with np.errstate(divide="ignore"):
        return np.where(z >= z_max, np.log(a["max"]), np.log(p_beam(z, z_hat, a, frac, lam)))


print(f"K = {K} beams, z_min = {z_min} m, z_max = {z_max} m, {60 / lds01['rotation_rpm']:.1f} s per scan")
for z_hat in (0.3, 1.0, 3.0):
    print(f"expected range {z_hat} m: sigma_hit = {100 * float(sigma_hit(z_hat)):.1f} cm")
z_grid = np.linspace(z_min, z_max, 20001)[:-1]
print(f"mass of the density below z_max for z_hat = 2 m: {np.trapezoid(p_beam(z_grid, 2.0), z_grid):.3f}")

In [ ]:
z = np.linspace(z_min, z_max - 1e-3, 3000)
fig, ax = plt.subplots(figsize=(6.5, 3.0))
for z_hat, color in ((1.0, BLUE), (3.0, VERMILLION)):
    ax.plot(z, p_beam(z, z_hat), color=color, lw=1.2, label=rf"$\hat z$ = {z_hat:g} m")
ax.annotate("", xy=(z_max, 10), xytext=(z_max, 0.003), arrowprops={"arrowstyle": "-|>", "color": GRAY, "lw": 1.5})
ax.text(z_max - 0.08, 18, rf"point mass $\alpha_{{\max}}$ = {alpha['max']} at $z_{{\max}}$ (arrow height not to scale)", ha="right", color=GRAY)
ax.set_yscale("log")
ax.set_ylim(0.003, 40)
ax.set_xlim(0, z_max + 0.1)
ax.set_xlabel("measured range $z$ (m)")
ax.set_ylabel(r"$p(z \mid \hat z)$ (1/m)")
ax.legend(loc="upper left")
fig.tight_layout()
plt.show()

*Density of one LDS-01 reading below $z_{\max}$ for expected ranges of 1 m and 3 m, on a log scale. The curves carry 96% of the probability. The other 4%, $\alpha_{\max}$, sits at $z_{\max}$, where the arrow stands for it.*

At 3 m the hit peak is three times wider than at 1 m, because the datasheet gives the precision as a percentage of the range. With these weights, the short-return part is larger than the random part below the peak, and beyond the peak only the random part remains.

In [ ]:
z_hat_c = 2.0
z = np.linspace(z_min, z_max - 1e-3, 2000)
fig, axes = plt.subplots(2, 2, figsize=(6.5, 4.4), sharex=True)
panels = [(p_hit(z, z_hat_c), r"$p_{\mathrm{hit}}$: measurement noise", BLUE),
          (p_short(z, z_hat_c), r"$p_{\mathrm{short}}$: unexpected objects", VERMILLION),
          (p_rand(z), r"$p_{\mathrm{rand}}$: random readings", GOLD), (None, r"$P_{\max}$: no return, unit point mass", GRAY)]
for ax, (p, title, color) in zip(axes.ravel(), panels):
    if p is None:
        ax.annotate("", xy=(z_max, 1.0), xytext=(z_max, 0), arrowprops={"arrowstyle": "-|>", "color": color, "lw": 2})
        ax.set_ylim(0, 1.15)
        ax.set_ylabel("probability")
    else:
        ax.plot(z, p, color=color, lw=1.3)
        ax.set_ylim(0, 1.15 * p.max())
        ax.set_ylabel("density (1/m)")
    if p is not None and "rand" not in title:  # only hit and short depend on z_hat
        ax.axvline(z_hat_c, color="black", ls=":", lw=0.8)
    ax.set_title(title)
    ax.set_xticks([0, z_hat_c, z_max], ["0", r"$\hat z$", r"$z_{\max}$"])
for ax in axes[1]:
    ax.set_xlabel("measured range $z$ (m)")
fig.tight_layout()
plt.show()

*The four components for one LDS-01 beam with an expected range $\hat z$ of 2 m, before mixing. $P_{\max}$ is a point mass at $z_{\max}$, not a density. Only $p_{\text{hit}}$ and $p_{\text{short}}$ depend on $\hat z$.*

The scan rate matters for a moving robot because a scan is not taken at one instant. The LDS-01 turns at 300 ± 10 rpm, so one scan takes about 0.2 s, and a robot driving at 0.2 m/s moves 4 cm during it. That is already more than $\sigma_{\text{hit}}$ at 1 m, 1.8 cm, and turning in place moves distant endpoints farther still. The model here treats a scan as taken from one pose $s_t$. A robot that drives fast, and every car, corrects each beam for the motion during the sweep, using the time at which the beam was measured.

No datasheet gives the mixture weights $\alpha$ or the rate $\lambda_{\text{short}}$. They depend on the scene: how many people walk through it, how much glass and dark furniture it holds, the weather. A darker surface returns a weaker echo and can be lost at a shorter range. The Hokuyo UST-10LX, for example, guarantees detection to 10 m on white card but only to 4 m on a 10% reflective surface ([spec sheet](https://www.ai.uni-bremen.de/wiki/_media/software/ust-10lx_specification.pdf)). How much that raises $\alpha_{\max}$ for the LDS-01 in a given room has to be measured there.

## A simulated scan

The [LIDAR section](/aiml-common/lectures/sensor-models/lidar/index#intensity-and-failure-modes) describes how readings go wrong. In the beam model, a return near the wall the map predicts is scored by the hit term, a return from a closer object such as a person or a chair by the short term, a beam with no return by the max term, and a reading in the valid range that none of these explains, from interference or multipath for example, by the random term.

The code below simulates one scan of a 2D LIDAR with a 7 m range in a room that contains each case, and labels every reading with the case that produced it. It generates the readings from the geometry of the scene, not by sampling the mixture. A wall return gets 3 cm of noise, a return from the person gets none, 3% of the beams are replaced by random readings, and every beam through the glass window or onto the dark bin is forced to return nothing. Real glass can also reflect or pass the beam, and a dark surface only weakens the echo. The forced rule stands in for those cases.

In [ ]:
# A room 8 m by 5 m. The map holds the walls only; the glass window is part of the top wall.
walls = np.array([[0, 0, 8, 0], [8, 0, 8, 5], [8, 5, 0, 5], [0, 5, 0, 0]], float)
glass = np.array([5.0, 5.0, 8.0, 5.0])  # beams through it are forced to return nothing
black_bin = (6.6, 1.3, 0.35)  # circle (x, y, radius); beams onto it are forced to return nothing
person = (3.4, 3.2, 0.25)  # circle, not in the map
sensor = np.array([2.0, 1.6])
z_max_sim, sigma_sim = 7.0, 0.03  # range limit and range noise of the simulated LIDAR


def ray_segment(p, u, seg):
    """Distance from p along the unit vector u to the segment (x0, y0, x1, y1), or inf."""
    a, e = seg[:2], seg[2:] - seg[:2]
    den = u[0] * e[1] - u[1] * e[0]
    if abs(den) < 1e-12:
        return np.inf
    w = a - p
    t = (w[0] * e[1] - w[1] * e[0]) / den
    k = (w[0] * u[1] - w[1] * u[0]) / den
    return t if t > 0 and 0 <= k <= 1 else np.inf


def ray_circle(p, u, circ):
    """Distance from p along the unit vector u to the circle (x, y, radius), or inf."""
    f = p - np.array(circ[:2])
    b = f @ u
    disc = b * b - (f @ f - circ[2] ** 2)
    if disc < 0:
        return np.inf
    t = -b - np.sqrt(disc)
    return t if t > 0 else np.inf


rng = np.random.default_rng(3)
phi_sim = np.deg2rad(np.arange(0, 360, 3))  # one beam every 3 degrees
z_hat_sim, z_sim, case = np.empty(len(phi_sim)), np.empty(len(phi_sim)), []
for k, ang in enumerate(phi_sim):
    u = np.array([np.cos(ang), np.sin(ang)])
    t_wall = min(ray_segment(sensor, u, s) for s in walls)
    z_hat_sim[k] = min(t_wall, z_max_sim)  # ray casting: distance to the first wall of the map
    through_glass = ray_segment(sensor, u, glass) <= t_wall + 1e-9
    t_bin, t_person = ray_circle(sensor, u, black_bin), ray_circle(sensor, u, person)
    if rng.random() < 0.03:
        case.append("random"); z_sim[k] = rng.uniform(0.3, z_max_sim)
    elif t_person < min(t_wall, t_bin):
        case.append("short"); z_sim[k] = t_person
    elif t_bin < t_wall or through_glass or t_wall > z_max_sim:
        case.append("max"); z_sim[k] = z_max_sim
    else:
        case.append("hit"); z_sim[k] = t_wall + rng.normal(0, sigma_sim)
case = np.array(case)
print({c: int(np.sum(case == c)) for c in ("hit", "short", "max", "random")})

In [ ]:
STYLE = {"hit": (GREEN, "hit: wall at the expected range"), "short": (VERMILLION, "short: unexpected object"),
         "max": (GRAY, "max range: no return"), "random": (GOLD, "random: unexplained return")}

fig, ax = plt.subplots(figsize=(6.5, 5.0))
for x0, y0, x1, y1 in walls:
    ax.plot([x0, x1], [y0, y1], color="black", lw=2)
ax.plot(glass[[0, 2]], glass[[1, 3]], color="#56b4e9", lw=4, label="glass window (no return)")
ax.add_patch(Circle(black_bin[:2], black_bin[2], color="#222222", label="dark bin (no return)"))
ax.add_patch(Circle(person[:2], person[2], color=PINK, label="person, not in the map"))
for key, (color, label) in STYLE.items():
    sel = case == key
    pts = sensor + z_sim[sel, None] * np.c_[np.cos(phi_sim[sel]), np.sin(phi_sim[sel])]
    for q in pts:
        ax.plot(*np.c_[sensor, q], color=color, lw=0.4, alpha=0.5)
    ax.plot(pts[:, 0], pts[:, 1], "o", ms=3, color=color, label=label)
ax.plot(*sensor, "s", color=BLUE, ms=8, label="LIDAR")
ax.set_aspect("equal")
ax.set_xlim(-0.4, 8.4)
ax.set_ylim(-0.4, 5.4)
ax.set_xlabel("$x$ (m)")
ax.set_ylabel("$y$ (m)")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=2)
fig.tight_layout()
plt.show()

*A simulated 2D scan, one beam every 3°. Hits on the walls (green), short returns from a person the map does not contain (orange), forced no-returns through the glass window and onto the dark bin (gray, drawn toward the 7 m maximum range and cut off at the edge of the plot), and a few random returns (gold).*

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.step(np.rad2deg(phi_sim), z_hat_sim, where="mid", color="black", lw=1, label=r"expected $\hat z_t^k$ (ray casting)")
for key, (color, label) in STYLE.items():
    sel = case == key
    ax.plot(np.rad2deg(phi_sim[sel]), z_sim[sel], "o", ms=3.5, color=color, label=f"measured: {key}")
ax.axhline(z_max_sim, color=GRAY, ls=":", lw=1)
ax.text(180, z_max_sim + 0.15, r"$z_{\max}$", color=GRAY, ha="center")
ax.set_xlim(0, 360)
ax.set_xticks(range(0, 361, 45))
ax.set_ylim(0, 7.8)
ax.set_xlabel(r"beam bearing $\phi_k$ (degrees)")
ax.set_ylabel("range (m)")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=3, fontsize=7)
fig.tight_layout()
plt.show()

*The simulated scan as ranges against bearing. The black line is the expected range $\hat z_t^k$ from ray casting in the map of walls. Most measured ranges $z_t^k$ sit on it; the person's beams read short, the beams through the glass and onto the bin read $z_{\max}$, and a few readings are random.*

## Estimating the parameters

The weights $\alpha$, the rate $\lambda_{\text{short}}$ and the width of $p_{\text{hit}}$ are estimated from readings $z_n$ whose expected ranges $\hat z_n$ are known, $\mathcal D = \{(z_n, \hat z_n)\}_{n=1}^N$. You get such data by recording scans at known poses in a room with a known map and ray casting each beam. Each $\hat z_n$ is the same quantity as $\hat z_t^k$, with one index running over all the beams of all the training scans.

<Note>
In the terms of supervised learning, $\hat z_n$ plays the role of the input $x$ and the reading $z_n$ the role of the target $y$, since the model is a conditional distribution $p(z \mid \hat z)$. Unlike a regression, nothing here learns to predict $z$ from $\hat z$: the hit term is already centered on $\hat z$. The fit learns only the shape of the error around that prediction, namely its width, the rate of early returns, and how often a beam returns nothing or a random range. In localization the roles change. The scan is observed, the filter computes $\hat z$ for each candidate pose by ray casting, and $p(z \mid \hat z)$ scores how well that pose explains the scan.
</Note>

If you knew which component produced each reading, every parameter would have a closed-form estimate. You do not, so *expectation maximization* (EM) alternates two steps, as in the algorithm `learn_intrinsic_parameters` (Thrun, Burgard and Fox, Section 6.3.2, Table 6.2).

The E-step computes, for every reading, the probability that each component produced it, its *responsibility*. A reading of exactly $z_{\max}$ can only come from the max component, so its responsibility $r_{n,\max}$ is 1. For a reading below $z_{\max}$,

$$
r_{n,i} = \frac{\alpha_i\, p_i(z_n \mid \hat z_n)}{\alpha_{\text{hit}}\, p_{\text{hit}}(z_n \mid \hat z_n) + \alpha_{\text{short}}\, p_{\text{short}}(z_n \mid \hat z_n) + \alpha_{\text{rand}}\, p_{\text{rand}}(z_n)}, \qquad i \in \{\text{hit}, \text{short}, \text{rand}\}.
$$

Table 6.2 as printed leaves the weights $\alpha_i$ out of this normalization. With them, each step is an EM step and can only increase the likelihood.

The M-step treats the responsibilities as soft labels and sets each parameter to its weighted maximum-likelihood value. The weights are the average responsibilities, $\alpha_i = \frac{1}{N} \sum_n r_{n,i}$. Table 6.2 sets the width from the weighted squared residuals. Here $\sigma_{\text{hit}} = c\,\hat z$ grows with range, so the same update applies to the relative residuals:

$$
c^2 = \frac{\sum_n r_{n,\text{hit}} \left( (z_n - \hat z_n) / \hat z_n \right)^2}{\sum_n r_{n,\text{hit}}}.
$$

This ignores the cut at $z_{\min}$ and $z_{\max}$ in $\eta_{\text{hit}}$, which is harmless when $\sigma_{\text{hit}}$ is a few centimeters. Table 6.2 sets $\lambda_{\text{short}} = \sum_n r_{n,\text{short}} / \sum_n r_{n,\text{short}}\, z_n$, which is the maximum-likelihood rate of an exponential with no upper cut. Here the cut at $\hat z$ matters, because a rate of 1/m spreads short readings over the whole interval. The code therefore maximizes the weighted log likelihood of the truncated exponential numerically.

The code below draws 20,000 readings from the model itself, with the parameters of the code above as the truth, at expected ranges between 0.5 m and 3.4 m. Because the true parameters are known, you can check that EM recovers them. EM starts far from them, with equal weights, $c = 0.05$ and $\lambda_{\text{short}} = 3$ per meter.

In [ ]:
from scipy.optimize import minimize_scalar

COMPONENTS = ("hit", "short", "max", "rand")


def sample_readings(z_hat, rng, a=alpha, frac=FRAC, lam=lam_short):
    """Draw one reading per expected range from the beam model. Returns the readings and the true components."""
    comp = rng.choice(4, size=len(z_hat), p=[a[c] for c in COMPONENTS])
    z = np.empty(len(z_hat))
    hit = comp == 0  # Gaussian, redrawn until it falls inside [z_min, z_max)
    z[hit] = z_hat[hit] + sigma_hit(z_hat[hit], frac) * rng.standard_normal(hit.sum())
    bad = hit & ((z < z_min) | (z >= z_max))
    while bad.any():
        z[bad] = z_hat[bad] + sigma_hit(z_hat[bad], frac) * rng.standard_normal(bad.sum())
        bad = hit & ((z < z_min) | (z >= z_max))
    short = comp == 1  # inverse CDF of the exponential cut to [z_min, z_hat]
    lo, hi = np.exp(-lam * z_min), np.exp(-lam * z_hat[short])
    z[short] = -np.log(lo - rng.random(short.sum()) * (lo - hi)) / lam
    z[comp == 2] = z_max
    z[comp == 3] = rng.uniform(z_min, z_max, (comp == 3).sum())
    return z, comp


def fit_beam_model(z, z_hat, n_iter=40):
    """EM for the weights, the relative width c of p_hit and lambda_short. Returns the estimates and the log likelihood per iteration."""
    a, frac, lam = dict.fromkeys(COMPONENTS, 0.25), 0.05, 3.0
    at_max = z >= z_max
    history = []
    for _ in range(n_iter):
        history.append(log_lik_beam(z, z_hat, a, frac, lam).sum())
        # E-step: responsibilities of hit, short, max and rand for every reading
        w = np.stack([a["hit"] * p_hit(z, z_hat, frac), a["short"] * p_short(z, z_hat, lam),
                      np.zeros_like(z), a["rand"] * p_rand(z)])
        w[:, at_max] = np.array([[0.0], [0.0], [1.0], [0.0]])  # a z_max reading is a max for certain
        r = w / w.sum(axis=0)
        # M-step: weighted maximum-likelihood estimates
        a = dict(zip(COMPONENTS, r.mean(axis=1)))
        frac = np.sqrt(np.sum(r[0] * ((z - z_hat) / z_hat) ** 2) / r[0].sum())
        neg_ll = lambda l: -np.sum(r[1] * np.log(np.maximum(p_short(z, z_hat, l), 1e-300)))
        lam = minimize_scalar(neg_ll, bounds=(0.01, 20.0), method="bounded").x
    lam_table62 = r[1].sum() / np.sum(r[1] * z)  # the update of Table 6.2, without the cut at z_hat
    return a, frac, lam, lam_table62, np.array(history)


rng_fit = np.random.default_rng(11)
z_hat_train = rng_fit.uniform(0.5, 3.4, 20_000)
z_train, comp_train = sample_readings(z_hat_train, rng_fit)
a_fit, frac_fit, lam_fit, lam_table62, ll_history = fit_beam_model(z_train, z_hat_train)

print("weight     true   fitted")
for c in COMPONENTS:
    print(f"{c:8s}  {alpha[c]:.3f}   {a_fit[c]:.3f}")
print(f"c         {FRAC:.4f}  {frac_fit:.4f}   (sigma_hit = c * z_hat)")
print(f"lambda    {lam_short:.3f}   {lam_fit:.3f}   (Table 6.2 update without the cut: {lam_table62:.3f})")
print(f"log likelihood: {ll_history[0]:.0f} at the start, {ll_history[-1]:.0f} after {len(ll_history)} iterations; "
      f"never decreased: {bool(np.all(np.diff(ll_history) >= -1e-6))}")

In [ ]:
z_hat_check = 2.0
z_check, _ = sample_readings(np.full(5000, z_hat_check), np.random.default_rng(12))
below = z_check[z_check < z_max]
fig, (ax_ll, ax_h) = plt.subplots(1, 2, figsize=(6.5, 2.9))
ax_ll.plot(np.arange(len(ll_history)), ll_history, color=BLUE, lw=1.3)
ax_ll.set_xlabel("EM iteration")
ax_ll.set_ylabel("log likelihood")
bins = np.linspace(z_min, z_max, 70)
ax_h.hist(below, bins=bins, weights=np.full(below.size, 1 / (z_check.size * (bins[1] - bins[0]))), color=GRAY, alpha=0.5,
          label="new readings")
zz = np.linspace(z_min, z_max - 1e-3, 2000)
ax_h.plot(zz, p_beam(zz, z_hat_check, a_fit, frac_fit, lam_fit), color=VERMILLION, lw=1.2, label="fitted model")
ax_h.set_yscale("log")
ax_h.set_ylim(0.005, 30)
ax_h.set_xlabel(r"measured range $z$ (m), $\hat z$ = 2 m")
ax_h.set_ylabel("density (1/m)")
ax_h.legend(loc="upper left", fontsize=7)
fig.tight_layout()
plt.show()

*Left: the log likelihood of the 20,000 readings at each EM iteration. Right: 5,000 new readings at an expected range of 2 m against the density of the fitted model; the readings at $z_{\max}$ are left out of the histogram.*

EM recovers the four weights to within 0.002 and the relative width $c$ to the printed digits, starting far from all of them, and the log likelihood rises at every iteration. The rate $\lambda_{\text{short}}$ comes out at 0.93 per meter against the true 1. It is the hardest parameter to pin down: only 8% of the readings are short, and a short reading near the sensor looks much like a random one. On the same responsibilities, the update of Table 6.2, which ignores the cut at $\hat z$, gives 1.35 per meter, 35% too high.

With real readings, the residuals also contain errors of the pose and the map, and the fit absorbs them into $\sigma_{\text{hit}}$ and the weights. Record the training scans at accurately known poses in a room whose map you trust, so that the estimates describe the sensor and the scene rather than those errors.

## References

- Thrun, S., Burgard, W., and Fox, D. (2005). *Probabilistic Robotics*. MIT Press. Section 6.3: the beam model (Eq. 6.11 to 6.13), the scan likelihood (Table 6.1) and the parameter learning (Table 6.2).
- ROBOTIS. [LDS-01 specifications](https://emanual.robotis.com/docs/en/platform/turtlebot3/appendix_lds_01/).
- Open Robotics. [REP 117, Informational Distance Measurements](https://www.ros.org/reps/rep-0117.html).